# Connect to Drive

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists("src"):
        !git clone https://github.com/HenriqueSchmitz/world-models-implementation temp_repo
        !mv temp_repo/* .
        !rm -rf temp_repo
        !sudo apt-get install -y swig
        %pip install -q -r requirements.txt

In [2]:
import os
from src.utils.colab import is_environment_colab_notebook
from src.utils.secrets import get_secret

project_folder = get_secret("worldModelsFolderPath") if is_environment_colab_notebook() else "./"
settings_path = os.path.join(project_folder, "settings.json")
settings_path = settings_path if os.path.exists(settings_path) else  "./settings.json"
world_model_folder = os.path.join(project_folder, "weights/worldmodel")
world_model_folder = world_model_folder if os.path.exists(world_model_folder) else  "./weights/worldmodel"
output_folder = os.path.join(project_folder, "weights/controller")
os.makedirs(output_folder, exist_ok=True)

# Training Settings

In [3]:
from src.utils.logging import get_logger
LOG_LEVEL = "INFO"
logger = get_logger(LOG_LEVEL)

2025-12-16 09:57:05 [INFO] Logger initialized.


In [4]:
import json

with open(settings_path, "r") as settings_file:
    settings = json.load(settings_file)
    REPRESENTATION_DIM = settings["vae"]["model"]["representation_dim"]
    RNN_HIDDEN_DIM = settings["world_model"]["model"]["hidden_dim"]
    ACTION_DIM = settings["controller"]["model"]["action_dim"]

    NUM_GENERATIONS = settings["controller"]["train"]["num_generations"]
    POPULATION_SIZE = settings["controller"]["train"]["population_size"]
    STEPS_PER_ROLLOUT = settings["controller"]["train"]["steps_per_rollout"]
    ROLLOUTS_PER_SOLUTION = settings["controller"]["train"]["rollouts_per_solution"]
    SIGMA_INIT = settings["controller"]["train"]["sigma_init"]

WANDB_PROJECT = "world-models-paper"
WANDB_RUN_NAME = "controller"

def log_settings():
    logger.info(f"REPRESENTATION_DIM: {REPRESENTATION_DIM}")
    logger.info(f"RNN_HIDDEN_DIM: {RNN_HIDDEN_DIM}")
    logger.info(f"ACTION_DIM: {ACTION_DIM}")
    logger.info("========================================")
    logger.info(f"NUM_GENERATIONS: {NUM_GENERATIONS}")
    logger.info(f"POPULATION_SIZE: {POPULATION_SIZE}")
    logger.info(f"STEPS_PER_ROLLOUT: {STEPS_PER_ROLLOUT}")
    logger.info(f"ROLLOUTS_PER_SOLUTION: {ROLLOUTS_PER_SOLUTION}")
    logger.info(f"SIGMA_INIT: {SIGMA_INIT}")
    logger.info("========================================")
    logger.info(f"WANDB_PROJECT: {WANDB_PROJECT}")
    logger.info(f"WANDB_RUN_NAME: {WANDB_RUN_NAME}")

log_settings()

2025-12-16 09:57:05 [INFO] REPRESENTATION_DIM: 32
2025-12-16 09:57:05 [INFO] RNN_HIDDEN_DIM: 256
2025-12-16 09:57:05 [INFO] ACTION_DIM: 3
2025-12-16 09:57:05 [INFO] ========================================
2025-12-16 09:57:05 [INFO] NUM_GENERATIONS: 1000
2025-12-16 09:57:05 [INFO] POPULATION_SIZE: 64
2025-12-16 09:57:05 [INFO] STEPS_PER_ROLLOUT: 1000
2025-12-16 09:57:05 [INFO] ROLLOUTS_PER_SOLUTION: 16
2025-12-16 09:57:05 [INFO] SIGMA_INIT: 0.1
2025-12-16 09:57:05 [INFO] ========================================
2025-12-16 09:57:05 [INFO] WANDB_PROJECT: world-models-paper
2025-12-16 09:57:05 [INFO] WANDB_RUN_NAME: controller


# Setup

In [5]:
from src.models.controller import Controller
from src.models.simulation_worldmodel import SimulationWorldModel
from src.training.controller_trainer import ControllerEvolutionaryTrainer
from src.utils.torch import get_device
from src.utils.secrets import get_secret

In [6]:
DEVICE = get_device()

2025-12-16 09:57:07 [INFO] Logger initialized.
2025-12-16 09:57:07 [INFO] Using device: mps:0


# Train

In [7]:
world_model_path = os.path.join(world_model_folder, f"model.pth")
simulation_worldmodel = SimulationWorldModel(worldmodel_path=world_model_path,
                                             settings_path=settings_path,
                                             device=DEVICE,
                                             batch_size=ROLLOUTS_PER_SOLUTION,
                                             logger=logger)

In [8]:
model = Controller(observation_dim=REPRESENTATION_DIM,
                   hidden_dim=RNN_HIDDEN_DIM,
                   action_dim=ACTION_DIM,
                   device=DEVICE)

In [9]:
try:
    wandb_setup = {
        "api_key": get_secret('wandbApiKey'),
        "project": WANDB_PROJECT,
        "run_name": WANDB_RUN_NAME,
        "config": {
            "generations": NUM_GENERATIONS,
            "population_size": POPULATION_SIZE,
            "steps_per_rollout": STEPS_PER_ROLLOUT,
            "rollouts_per_solution": ROLLOUTS_PER_SOLUTION,
            "sigma_init": SIGMA_INIT,
        }
    }
except Exception as e:
    logger.error(f"Ignoring wandb logging, could not retrieve wandbApiKey secret. Error: {str(e)}")
    wandb_setup = None

In [10]:
trainer = ControllerEvolutionaryTrainer(model=model,
                                        weights_folder=output_folder,
                                        num_generations=NUM_GENERATIONS,
                                        population_size=POPULATION_SIZE,
                                        simulation_world_model=simulation_worldmodel,
                                        steps_per_rollout=STEPS_PER_ROLLOUT,
                                        rollouts_per_solution=ROLLOUTS_PER_SOLUTION,
                                        sigma_init=SIGMA_INIT,
                                        load_checkpoint=True,
                                        device=DEVICE,
                                        wandb_setup=wandb_setup,
                                        logger=logger)
                                        

2025-12-16 09:57:07 [INFO] No existing checkpoints found. Starting from scratch.


(32_w,64)-aCMA-ES (mu_w=17.6,w_1=11%) in dimension 578 (seed=969324, Tue Dec 16 09:57:08 2025)


In [11]:
# Repeat logging of important information so it is available on wandb run
log_settings()
get_device(logger)

2025-12-16 09:57:08 [INFO] REPRESENTATION_DIM: 32
2025-12-16 09:57:08 [INFO] RNN_HIDDEN_DIM: 256
2025-12-16 09:57:08 [INFO] ACTION_DIM: 3
2025-12-16 09:57:08 [INFO] ========================================
2025-12-16 09:57:08 [INFO] NUM_GENERATIONS: 1000
2025-12-16 09:57:08 [INFO] POPULATION_SIZE: 64
2025-12-16 09:57:08 [INFO] STEPS_PER_ROLLOUT: 1000
2025-12-16 09:57:08 [INFO] ROLLOUTS_PER_SOLUTION: 16
2025-12-16 09:57:08 [INFO] SIGMA_INIT: 0.1
2025-12-16 09:57:08 [INFO] ========================================
2025-12-16 09:57:08 [INFO] WANDB_PROJECT: world-models-paper
2025-12-16 09:57:08 [INFO] WANDB_RUN_NAME: controller
2025-12-16 09:57:08 [INFO] Using device: mps:0


device(type='mps', index=0)

In [ ]:
trainer.train()

Epoch:   0%|          | 0/1000 [00:00<?, ?epoch/s]

Train Epoch 1:   0%|          | 0/64 [00:00<?, ?batch/s]

2025-12-16 09:57:33 [INFO] Epoch 1 Training Average Reward: -99.0663
2025-12-16 09:57:33 [INFO] Epoch 1 Training Max Reward: -91.6653
2025-12-16 09:57:57 [INFO] Epoch 2 Training Average Reward: -97.8772
2025-12-16 09:57:57 [INFO] Epoch 2 Training Max Reward: -90.4058
2025-12-16 09:58:23 [INFO] Epoch 3 Training Average Reward: -96.6874
2025-12-16 09:58:23 [INFO] Epoch 3 Training Max Reward: -80.1889
2025-12-16 09:58:47 [INFO] Epoch 4 Training Average Reward: -93.7511
2025-12-16 09:58:47 [INFO] Epoch 4 Training Max Reward: -76.4003
2025-12-16 09:59:13 [INFO] Epoch 5 Training Average Reward: -90.5701
2025-12-16 09:59:13 [INFO] Epoch 5 Training Max Reward: -52.4930
2025-12-16 09:59:37 [INFO] Epoch 6 Training Average Reward: -85.9544
2025-12-16 09:59:37 [INFO] Epoch 6 Training Max Reward: -61.5057
2025-12-16 10:00:02 [INFO] Epoch 7 Training Average Reward: -68.4277
2025-12-16 10:00:02 [INFO] Epoch 7 Training Max Reward: 345.2092
2025-12-16 10:00:28 [INFO] Epoch 8 Training Average Reward: -7